In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool, Tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown
from exa_py import Exa

In [ ]:
load_dotenv(override=True)

In [ ]:
import os

api_key = os.getenv('OPENAI_API_KEY')

if api_key:
    print(f"OpenAi API Key exists and begins {api_key[:8]}")
else:
    print("API Key not set - please head to the troubleshooting guide in the setup folder")

print(os.getenv('OPENAI_BASE_URL'))
print(os.getenv('OPENAI_API_KEY'))

In [ ]:
#Instead of using WebSearchTool, using Exa search tool which is free
exa = Exa(api_key=os.environ["EXA_API_KEY"])

@function_tool
def exa_search_tool(query: str) -> str:
    """Search the web for latest information"""
    
    results = exa.search_and_contents(query, num_results=3)
    
    return "\n\n".join(
        f"{r.title}\n{r.text[:300]}"
        for r in results.results
    )

In [ ]:
instructions= "You are an research assistant. Given a search query, gather relevant \
web information and generate a compact summary. \
\
Requirements: \
Length: 3-4 paragraphs, under 300 words \
Content: only essential insights and main points \
Style: concise, compressed, and information-dense \
\
Rules: \
Exclude filler, repetition, and low-value details \
Do not add explanations, opinions, or extra commentary \
Output only the final summary"

research_agent = Agent(
    name="Research Assistant", 
    instructions=instructions,
    tools=[exa_search_tool],
    model="gpt-oss-20b",
    model_settings=ModelSettings(tool_choice="required"))

In [ ]:
message = "What are Latest AI Agent frameworks"

with trace("Search"):
    result = await Runner.run(research_agent, message)

display(Markdown(result.final_output))

In [ ]:
TOTAL_SEARCHES = 3

instructions1 = f"You are an research assistant. Given a search query, gather relevant \
web information and come up with a set of web searches to perform to best answer the query. \
Output {TOTAL_SEARCHES} terms to query for. \
\
Requirements: \
Length: 3-4 paragraphs, under 300 words \
Content: only essential insights and main points \
Style: concise, compressed, and information-dense \
\
Rules: \
Exclude filler, repetition, and low-value details \
Do not add explanations, opinions, or extra commentary \
Output only the final summary"

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this is important to the query.")

    query: str = Field(description="The search term to use for the web search.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform the best answer the query.")

planner_agent = Agent(
    name="PlannerAgent",
    instructions=instructions1,
    model="gpt-oss-20b",
    output_type=WebSearchPlan,
)

In [ ]:
message = "What are Latest AI Agent frameworks"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

In [ ]:
@function_tool
def send_email(subject: str, body: str) -> Dict[str, str]:
    """Send email to all prospect Clients"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("cksharpa@gmail.com")  # Change to your verified sender
    to_email = To("<receiver>@gmail.com")  # Change to your recipient
    content = Content("text/html", body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return  "success"

In [ ]:
instructions ="You are responsible for formatting and sending emails. \
Given an email body, first generate an appropriate subject using the subject_writer tool. \
Then convert the body into HTML using the html_converter tool. \
Finally, send the email using the send_html_email tool with the generated subject and HTML content."

email_agent = Agent(name="Email Manager", instructions=instructions, 
    tools=[send_email], model="gpt-oss-20b",)

In [ ]:
INSTRUCTIONS = (
    "You are a Senior Researcher responsible for producing a comprehensive and well-structured report."
    "You will be provided with a primary research query preliminary research findings generated by a research assistant.\n"
    "You should review the research query thoroughly, Examine the provided preliminary research for key "
    "insights and gaps. Develop a clear, logical outline that defines the structure and flow of the report."
    "Include major sections, subsections and key points to be covered. Then, Expand the outline into a detailed, conhesive report.\n"
    "Ensure smooth transitions and strong narrative flow synthesize and refine the provided research."
    "The final output should be clear, concise, and professional in markdown format.It should be lengthy and detailed document of "
    "minimum 1000 words (approximately 5–10 pages). Focus on key insights and meaningful analysis. "
    "Avoid unnecessary filler or repetition."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


senior_research_agent = Agent(
    name="Senior Research Agent",
    instructions=INSTRUCTIONS,
    model="gpt-oss-20b",
    output_type=ReportData,
)

In [ ]:
#Plan and Execute the search using planner_agent and research_agent
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(research_agent, input)
    return result.final_output

In [ ]:
#Write a report and send email
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(senior_research_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

In [ ]:
query ="Latest AI Agent frameworks"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report) 